# La Grulla Blanca — NPC Behavior Benchmark v0.1\n\nNotebook para ejecutar el laboratorio canónico en Google Colab sin modificar producción.\n\n**Importante:** CANON y perfiles/políticas experimentales permanecen separados. Los resultados no asignan automáticamente una IA definitiva.

In [ ]:
REPO = 'https://github.com/Shein25/La-Grulla-Blanca.git'\nBRANCH = 'experiment/npc-canonical-behavior-lab-v0.1'\nTIER = 'standard'   # smoke | standard | deep | million\nNPC = 'all'        # all | gao_shun | pei_luo | jiang_rui\nSEED = 1337\nCUSTOM_RUNS = None  # ej. 2_000_000; si se define, reemplaza TIER\n\nprint('Configuración:', {'tier':TIER,'npc':NPC,'seed':SEED,'custom_runs':CUSTOM_RUNS})

In [ ]:
import os, subprocess, pathlib, shutil\n\nroot = pathlib.Path('/content/La-Grulla-Blanca')\nif root.exists():\n    shutil.rmtree(root)\nsubprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(root)], check=True)\nlab = root/'experimentos'/'npc'/'canonical-behavior-lab-v0.1'\nos.chdir(lab)\nprint('LAB:', lab)\nsubprocess.run(['node','--version'], check=True)

In [ ]:
# Verificación del benchmark antes de una corrida grande\nsubprocess.run(['node','benchmark/test-benchmark.mjs'], check=True)

In [ ]:
out = '/content/npc_behavior_benchmark.json'\ncmd = ['node','benchmark/run-benchmark.mjs','--tier',TIER,'--seed',str(SEED),'--npc',NPC,'--out',out]\nif CUSTOM_RUNS is not None:\n    cmd += ['--runs',str(int(CUSTOM_RUNS))]\nprint('Ejecutando:', ' '.join(cmd))\nsubprocess.run(cmd, check=True)\nprint('Resultado:', out)

In [ ]:
import json, pandas as pd\nwith open('/content/npc_behavior_benchmark.json','r',encoding='utf-8') as f:\n    data=json.load(f)\n\nrows=[]\nfor npc_name,r in data['results'].items():\n    m=r['metrics']\n    row={'npc':npc_name,'mode':r['mode'],'runs':m['runs'],'invalidIntents':m.get('invalidIntents',0)}\n    if npc_name=='jiang_rui':\n        row.update({\n            'allThreeAgreementPct':m['agreementPct']['allThree'],\n            'fsmBtAgreementPct':m['agreementPct']['fsm_bt'],\n            'fsmUtilityAgreementPct':m['agreementPct']['fsm_utility'],\n            'btUtilityAgreementPct':m['agreementPct']['bt_utility'],\n            'invalidContexts':m['invalidContexts']\n        })\n    else:\n        row.update({\n            'sequenceAgreement':m['sequenceAgreement'],\n            'sequenceDisagreement':m['sequenceDisagreement'],\n            'btPreemptions':m['btPreemptions']\n        })\n    rows.append(row)\n\ndisplay(pd.DataFrame(rows))\nprint('Digest reproducible:', data['digest'])

In [ ]:
# Superficie de desacuerdos de Jiang Rui\njr=data['results'].get('jiang_rui')\nif jr:\n    display(pd.DataFrame(jr['examples']))\n    print('Distribución Utility:', jr['metrics']['utilityActions'])\n    print('Buckets:', json.dumps(jr['metrics']['buckets'], indent=2, ensure_ascii=False))

## Escalado sugerido\n\n- `smoke`: comprobar que el notebook/harness funciona.\n- `standard`: comparación normal.\n- `deep`: buscar fronteras menos frecuentes.\n- `million`: stress / alta resolución, preferentemente sobre un NPC concreto.\n- `CUSTOM_RUNS`: permite 2M, 5M, 10M, etc., sólo cuando la pregunta lo justifique.\n\nPara millones de ejecuciones sobre todos los NPC conviene primero mirar la corrida `standard` y decidir dónde hace falta más resolución.